# CS781 Phase 3 — Rhetorical Role Prediction (Colab TPU starter pipeline)

**Runtime target:** Google Colab TPU runtime, where available.

**Important:** TPU support is not a one-line switch. This notebook provides TPU-aware setup and a PyTorch/XLA detection path, but the current generic training code and architecture-specific Phase 1 loaders are not guaranteed to run on TPU unchanged. Start with the runtime/device check and a tiny smoke test before attempting full training.

- Uses only your own Phase 1 Legal Language Model checkpoints; it does not download unrelated pretrained model weights.
- Mount Google Drive for checkpoints and persistent outputs.
- The Phase 3 dataset may be gated on Hugging Face; you need access and a Hugging Face token if downloading it.
- The four architecture loaders remain placeholders until connected to your actual Phase 1 implementations.


## Colab setup checklist\n1. In Colab choose **Runtime → Change runtime type → TPU** (if available). This cannot be enabled by notebook code itself.\n2. Run the install and imports cells; approve the Google Drive mount prompt.\n3. Put your Phase 1 checkpoint and tokenizer in Drive and fill in their paths.\n4. Configure the correct loader for one architecture. The existing template is not a complete loader.\n5. Run data validation, tokenization, then the smoke test. Start with batch size 2 and gradient accumulation 8.\n\n**Important limitation:** this notebook is made Colab/TPU-aware, but it does not automatically implement the Transformer/Mamba/RMKV/RetNet loaders, LORMA, or the full KD training loop. Those must be implemented before the required experiments can run.

**TPU-specific caveat:** TPU execution requires a compatible PyTorch/XLA install and XLA-aware optimizer stepping. The existing generic training loop uses standard PyTorch patterns and must be adapted before full TPU training. A TPU runtime may also be slower than a T4 for small workloads due to compilation overhead.

## 0. What this notebook covers

1. Access and inspect the Phase 3 dataset.
2. Validate sentence/label alignment and the official 13-label vocabulary.
3. Create a document-level train/dev split only if official dev files are unavailable (the supplied CL/IT dev files are preferred).
4. Build sentence-only and neighboring-sentence-context examples.
5. Train a sequence classifier on a Phase 1 model, with a classification head over target-sentence token representations.
6. Compare full fine-tuning and LoRA; provide a clearly marked hook for the compulsory LORMA implementation and a teacher→student distillation run.
7. Log per-domain and overall macro-F1, per-class F1, runtime, and best checkpoint.
8. Generate a `prediction.json` preserving test document IDs and sentence order.

**Not automatically solved by a generic notebook:** loading your Mamba, RMKV, RetNet, and Transformer checkpoints. Register the real model classes and tokenizer for each architecture in Section 6. The code stops if a slot is not configured rather than guessing.

## Install dependencies

# Run in a fresh Colab TPU runtime. PyTorch/XLA versions must match the Colab runtime.
%pip -q install transformers peft scikit-learn pandas numpy huggingface_hub safetensors tokenizers
# If torch_xla is missing, use the install command recommended for your current Colab TPU runtime:
# %pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html


In [ ]:
## 2. Imports, Drive setup, and TPU detection


import os, json, math, time, random, copy, gc, inspect, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from huggingface_hub import snapshot_download, HfApi
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from google.colab import drive
drive.mount("/content/drive")

SEED = 781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# TPU needs PyTorch/XLA. CPU fallback is provided for data checks only.
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
    TPU_AVAILABLE = True
    device = xm.xla_device()
    print("PyTorch/XLA:", torch_xla.__version__)
    print("TPU device:", device)
except Exception as e:
    TPU_AVAILABLE = False
    xm = None
    xr = None
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch/XLA TPU unavailable:", repr(e))
    print("Detected fallback device:", device)
    print("Use Runtime → Change runtime type → TPU, then install a compatible torch_xla build.")

WORK_DIR = Path("/content/drive/MyDrive/CS781/phase3_rr")
RUNS_DIR = WORK_DIR / "runs"
PRED_DIR = WORK_DIR / "predictions"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
print("outputs:", WORK_DIR)


In [ ]:
import os, json, math, time, random, copy, gc, inspect, warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from huggingface_hub import snapshot_download, HfApi
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

# Mount Google Drive so checkpoints and results persist across Colab sessions.
from google.colab import drive
drive.mount("/content/drive")

SEED = 781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1024**3:.1f} GiB")
    if "T4" not in torch.cuda.get_device_name(0):
        print("NOTE: TPU runtime detection is handled in the imports cell.")

# Dataset repository. If you have already downloaded Phase_3_RR to Drive, set LOCAL_DATA_ROOT in the next cell.
HF_DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATASET_SUBDIR = "Phase_3_RR"
HF_TOKEN = os.environ.get("HF_TOKEN", None)  # Better: set this in Colab Secrets; do not hard-code a token.

# Keep all experiment outputs in Drive so they survive runtime disconnects.
WORK_DIR = Path("/content/drive/MyDrive/CS781/phase3_rr")
RUNS_DIR = WORK_DIR / "runs"
PRED_DIR = WORK_DIR / "predictions"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
print("outputs:", WORK_DIR)


## 3. Download / locate the dataset


In [ ]:
# Option A (recommended if already downloaded): set this to the Phase_3_RR folder in Google Drive.
# Example: "/content/drive/MyDrive/CS781/Phase_3_RR"
# Option B: leave None to download from Hugging Face (requires access; gated/private repos need a token).
LOCAL_DATA_ROOT = None

def locate_dataset_root(root):
    root = Path(root)
    candidates = [root, root / DATASET_SUBDIR]
    for p in candidates:
        if (p / "Train").exists() and (p / "Test").exists():
            return p
    for p in candidates:
        if (p / "train").exists() and (p / "test").exists():
            return p
    raise FileNotFoundError(f"Could not find Train/Test folders under {root}")

if LOCAL_DATA_ROOT:
    DATA_ROOT = locate_dataset_root(LOCAL_DATA_ROOT)
else:
    try:
        downloaded = snapshot_download(
            repo_id=HF_DATASET_ID,
            repo_type="dataset",
            token=HF_TOKEN,
            allow_patterns=[f"{DATASET_SUBDIR}/**", "*.json", "**/*.json"]
        )
        DATA_ROOT = locate_dataset_root(downloaded)
    except Exception as e:
        print("Dataset download failed. Common causes: gated/private dataset or missing Hugging Face access.")
        print("If the dataset is already in Drive, set LOCAL_DATA_ROOT above. Otherwise accept/request access and configure HF_TOKEN in Colab Secrets.")
        print("Original error:", repr(e))
        raise

print("DATA_ROOT =", DATA_ROOT)
print("Train/dev files:", sorted(str(p.relative_to(DATA_ROOT)) for p in DATA_ROOT.rglob("*")
                                  if p.is_file() and ("train" in p.name.lower() or "dev" in p.name.lower())))


## 4. Load JSON safely and validate the dataset

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def find_file(root, filename):
    hits = list(Path(root).rglob(filename))
    if not hits:
        raise FileNotFoundError(f"Missing {filename} under {root}")
    if len(hits) > 1:
        print(f"Multiple matches for {filename}; using", hits[0])
    return hits[0]

label_path = find_file(DATA_ROOT, "label_vocab.json")
label_obj = read_json(label_path)
if isinstance(label_obj, dict):
    if "labels" in label_obj and isinstance(label_obj["labels"], list):
        LABELS = list(label_obj["labels"])
    else:
        # If vocabulary is label->id, order by ID; otherwise preserve keys.
        try:
            LABELS = [k for k,v in sorted(label_obj.items(), key=lambda kv: int(kv[1]))]
        except Exception:
            LABELS = list(label_obj.keys())
elif isinstance(label_obj, list):
    LABELS = list(label_obj)
else:
    raise ValueError(f"Unrecognized label_vocab format: {type(label_obj)}")

EXPECTED_LABELS = [
    "Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent",
    "PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled",
    "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"
]
print("Official labels:", LABELS)
if set(LABELS) != set(EXPECTED_LABELS):
    print("WARNING: labels differ from expected list. Use label_vocab.json as source of truth.")
label2id = {label:i for i,label in enumerate(LABELS)}
id2label = {i:label for label,i in label2id.items()}

def normalize_doc(doc, require_labels):
    # Expected schema: {"id": "...", "text": [sentence, ...], "labels": [label, ...]}
    if not isinstance(doc, dict):
        raise TypeError(f"Document must be dict, got {type(doc)}")
    did = str(doc.get("id", doc.get("doc_id", doc.get("document_id", ""))))
    texts = doc.get("text", doc.get("sentences", None))
    labs = doc.get("labels", None)
    if not did:
        raise ValueError(f"Missing document ID; keys={list(doc)}")
    if not isinstance(texts, list):
        raise ValueError(f"{did}: expected list under 'text' or 'sentences'")
    texts = [str(x) for x in texts]
    if require_labels:
        if not isinstance(labs, list):
            raise ValueError(f"{did}: missing labels list")
        if len(texts) != len(labs):
            raise ValueError(f"{did}: {len(texts)} sentences vs {len(labs)} labels")
        unknown = sorted(set(map(str, labs)) - set(LABELS))
        if unknown:
            raise ValueError(f"{did}: labels absent from label_vocab.json: {unknown}")
        labs = list(map(str, labs))
    return {"id": did, "text": texts, "labels": labs}

def load_split(filename, require_labels):
    obj = read_json(find_file(DATA_ROOT, filename))
    # Usually a list of docs; tolerate dict keyed by ID.
    if isinstance(obj, dict):
        if any(k in obj for k in ("text", "sentences")):
            raw = [obj]
        else:
            raw = []
            for did, val in obj.items():
                val = dict(val)
                val.setdefault("id", did)
                raw.append(val)
    elif isinstance(obj, list):
        raw = obj
    else:
        raise ValueError(f"Unexpected JSON root for {filename}: {type(obj)}")
    docs = [normalize_doc(x, require_labels) for x in raw]
    ids = [d["id"] for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate document IDs in {filename}")
    return docs

CL_TRAIN = load_split("CL_train.json", True)
CL_DEV = load_split("CL_dev.json", True)
IT_TRAIN = load_split("IT_train.json", True)
IT_DEV = load_split("IT_dev.json", True)
CL_TEST = load_split("CL_test.json", False)
IT_TEST = load_split("IT_test.json", False)

for name, docs in [("CL train",CL_TRAIN),("CL dev",CL_DEV),("IT train",IT_TRAIN),("IT dev",IT_DEV),("CL test",CL_TEST),("IT test",IT_TEST)]:
    n_docs = len(docs); n_sent = sum(len(d["text"]) for d in docs)
    print(f"{name:8s}: {n_docs:5d} docs | {n_sent:8d} sentences")

def label_counts(docs):
    return Counter(y for d in docs for y in d["labels"])
counts = label_counts(CL_TRAIN) + label_counts(IT_TRAIN)
print("\nTraining label counts:")
display(pd.DataFrame({"label": LABELS, "count": [counts[l] for l in LABELS]}).sort_values("count"))

## 5. Context construction

The target sentence is marked explicitly, with one previous and one next sentence by default. This is a local-context baseline, not leakage: at inference time those neighboring sentences are available in the same test document. We do **not** feed neighboring gold labels into the model.

In [ ]:
CONTEXT_RADIUS = 1  # 0 = sentence-only; 1 = previous + target + next
MAX_LENGTH = 512    # adjust to each Phase 1 architecture's supported context
TARGET_MARKERS = ("<TARGET>", "</TARGET>")

def make_input(sentences, i, radius=1):
    left = max(0, i-radius)
    right = min(len(sentences), i+radius+1)
    pieces = []
    for j in range(left, right):
        if j == i:
            pieces.append(f"<TARGET> {sentences[j]} </TARGET>")
        else:
            pieces.append(sentences[j])
    return " ".join(pieces)

def flatten_docs(docs, radius=1, with_labels=True):
    rows = []
    for d in docs:
        for i, sent in enumerate(d["text"]):
            rows.append({
                "doc_id": d["id"],
                "sentence_index": i,
                "target_text": sent,
                "input_text": make_input(d["text"], i, radius),
                "label": d["labels"][i] if with_labels else None
            })
    return rows

train_rows = flatten_docs(CL_TRAIN, CONTEXT_RADIUS) + flatten_docs(IT_TRAIN, CONTEXT_RADIUS)
dev_cl_rows = flatten_docs(CL_DEV, CONTEXT_RADIUS)
dev_it_rows = flatten_docs(IT_DEV, CONTEXT_RADIUS)
print("Train examples:", len(train_rows), "| CL dev:", len(dev_cl_rows), "| IT dev:", len(dev_it_rows))
print(train_rows[0])

## 6. Register your four Phase 1 architectures

This is the one project-specific step that cannot safely be guessed from the transcript. For each architecture, supply:
- the exact Phase 1 checkpoint path,
- its model class/loader,
- its tokenizer (use the Phase 1 tokenizer, not a newly downloaded tokenizer),
- a function that returns contextual token hidden states shaped `[batch, seq, hidden]`.

The generic adapter below expects a Hugging Face-style encoder/backbone that returns `last_hidden_state` or a tuple whose first element is hidden states. For a causal GPT-2 model, use `GPT2Model` (not the LM head). For custom Mamba/RMKV/RetNet implementations, write a small wrapper in `load_backbone_fn` that returns hidden states with the same shape.

**Do not point every slot to the same checkpoint.** Each experiment must begin from the correct Phase 1 checkpoint. If the architecture's implementation is unavailable, mark it unavailable rather than substituting another model.

In [ ]:
# Add paths to your own Phase 1 checkpoints stored in Google Drive.
# IMPORTANT: loader entries remain None until you define the matching architecture-specific loader below.
# Example checkpoint path: "/content/drive/MyDrive/CS781/phase1/checkpoint_latest.pt"
PHASE1 = {
    "transformer": {
        "checkpoint": None,
        "tokenizer_json": None,
        "loader": None,  # load_transformer_backbone(checkpoint_path) -> nn.Module
    },
    "mamba": {"checkpoint": None, "tokenizer_json": None, "loader": None},
    "rmkv": {"checkpoint": None, "tokenizer_json": None, "loader": None},
    "retnet": {"checkpoint": None, "tokenizer_json": None, "loader": None},
}

def inspect_registry(registry=PHASE1):
    for name, cfg in registry.items():
        ok = bool(cfg.get("checkpoint")) and Path(cfg["checkpoint"]).exists() and callable(cfg.get("loader"))
        print(f"{name:12s}: {'READY' if ok else 'NOT CONFIGURED'}")
        if cfg.get("checkpoint") and not Path(cfg["checkpoint"]).exists():
            print("  missing checkpoint:", cfg["checkpoint"])
inspect_registry()


## 7. Backbone + sentence classifier

We pool the hidden states that correspond to the marked target sentence. To avoid depending on special-token IDs that may not exist in the original tokenizer, this implementation uses the target text's token span in the full context string. If your tokenizer/model wrapper cannot provide offsets, use the simpler CLS/last-token pooling variant in your architecture adapter.

The model loader should return a module whose `forward(input_ids, attention_mask)` returns hidden states or a tuple beginning with hidden states.

In [ ]:
class RhetoricalClassifier(nn.Module):
    def __init__(self, backbone, hidden_size, num_labels, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, target_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(out, "last_hidden_state"):
            h = out.last_hidden_state
        elif isinstance(out, (tuple, list)):
            h = out[0]
        elif torch.is_tensor(out):
            h = out
        else:
            raise TypeError(f"Unsupported backbone output: {type(out)}")
        mask = target_mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        return self.classifier(self.dropout(pooled))

def infer_hidden_size(backbone):
    for attr in ("config",):
        cfg = getattr(backbone, attr, None)
        if cfg is not None:
            for key in ("hidden_size", "n_embd", "d_model", "dim"):
                if hasattr(cfg, key):
                    return int(getattr(cfg, key))
    raise ValueError("Cannot infer hidden size. Return backbone.config.hidden_size/n_embd/d_model or set it manually.")

## 8. Tokenization with target masks

This cell uses a fast Hugging Face tokenizer that returns offsets. If your Phase 1 tokenizer is a `tokenizers.Tokenizer` JSON, wrap it in a small adapter or convert it to a compatible `PreTrainedTokenizerFast` using your existing tokenizer configuration. Do not download a tokenizer from an unrelated model.

In [ ]:
# Path to a Hugging Face-compatible local tokenizer directory in Drive.
# If you only have tokenizer.json, it may need a PreTrainedTokenizerFast wrapper using the correct special-token IDs.
PHASE1_TOKENIZER_PATH = None  # e.g. "/content/drive/MyDrive/CS781/phase1/tokenizer"

if PHASE1_TOKENIZER_PATH:
    tokenizer = AutoTokenizer.from_pretrained(PHASE1_TOKENIZER_PATH, use_fast=True, local_files_only=True)
else:
    tokenizer = None
    print("Set PHASE1_TOKENIZER_PATH to your own Phase 1 tokenizer before running tokenization.")

class RRDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=512, with_labels=True):
        self.rows, self.tokenizer, self.max_length, self.with_labels = rows, tokenizer, max_length, with_labels
        self.items = []
        for row in rows:
            text = row["input_text"]
            enc = tokenizer(text, truncation=True, max_length=max_length, padding="max_length",
                            return_offsets_mapping=True, return_attention_mask=True)
            offsets = enc.pop("offset_mapping")
            a = text.find("<TARGET>") + len("<TARGET>")
            b = text.find("</TARGET>")
            if a < len("<TARGET>") or b < 0 or b <= a:
                raise ValueError("Target markers missing from input")
            target_mask = [int(end > a and start < b and end > start) for start, end in offsets]
            if not any(target_mask):
                raise ValueError(f"Target sentence truncated away: {row['doc_id']} idx={row['sentence_index']}")
            self.items.append({
                "input_ids": enc["input_ids"],
                "attention_mask": enc["attention_mask"],
                "target_mask": target_mask,
                "label_id": label2id[row["label"]] if with_labels else -100,
                "doc_id": row["doc_id"],
                "sentence_index": row["sentence_index"],
                "target_text": row["target_text"],
            })
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        x = self.items[i]
        return {k: torch.tensor(x[k], dtype=torch.long) for k in ("input_ids","attention_mask","target_mask","label_id")}

def make_loader(rows, tokenizer, batch_size=2, shuffle=False, with_labels=True):
    ds = RRDataset(rows, tokenizer, MAX_LENGTH, with_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=(device.type=="cuda")), ds


## 9. Metrics and evaluation

Use macro-F1 over the full official label list. This is important because rare labels matter just as much as frequent labels in the primary metric. CL and IT are reported separately, as well as combined.

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "macro_f1": f1_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0),
    }

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    ys, ps = [], []
    for batch in loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch["target_mask"])
        ys.extend(batch["label_id"].cpu().tolist())
        ps.extend(logits.argmax(-1).cpu().tolist())
    return compute_metrics(ys, ps), ys, ps

def per_class_report(y_true, y_pred):
    return pd.DataFrame(classification_report(
        y_true, y_pred, labels=list(range(len(LABELS))), target_names=LABELS,
        output_dict=True, zero_division=0
    )).T

## 10. Training loop (full fine-tuning or frozen-backbone baseline)

The baseline starts from the registered Phase 1 checkpoint. Use conservative learning rates and early selection by development macro-F1. Checkpoint files are written only under this run's directory.

In [ ]:
def train_one_run(run_name, model, train_loader, dev_loaders, epochs=5, lr_backbone=2e-5,
                  lr_head=1e-3, weight_decay=0.01, grad_accum=1, freeze_backbone=False,
                  max_grad_norm=1.0, patience=2):
    run_dir = RUNS_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    if freeze_backbone:
        for p in model.backbone.parameters(): p.requires_grad = False
    else:
        for p in model.backbone.parameters(): p.requires_grad = True
    head_params = list(model.classifier.parameters())
    body_params = [p for p in model.backbone.parameters() if p.requires_grad]
    groups = [{"params": head_params, "lr": lr_head}]
    if body_params: groups.append({"params": body_params, "lr": lr_backbone})
    optimizer = torch.optim.AdamW(groups, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_steps = max(1, steps_per_epoch * epochs)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06*total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type=="cuda"))
    best_f1, best_epoch, bad_epochs = -1.0, 0, 0
    history = []
    model.to(device)
    for epoch in range(1, epochs+1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        losses = []
        t0 = time.time()
        for step, batch in enumerate(train_loader, 1):
            batch = {k:v.to(device) for k,v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type=="cuda")):
                logits = model(batch["input_ids"], batch["attention_mask"], batch["target_mask"])
                loss = nn.functional.cross_entropy(logits.float(), batch["label_id"])
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch {epoch}, step {step}")
            scaler.scale(loss / grad_accum).backward()
            losses.append(float(loss.detach().cpu()))
            if step % grad_accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(set_to_none=True); scheduler.step()
        domain_metrics = {}
        combined_y, combined_p = [], []
        for domain, loader in dev_loaders.items():
            met, y, p = evaluate_model(model, loader)
            domain_metrics[domain] = met
            combined_y += y; combined_p += p
        combined = compute_metrics(combined_y, combined_p)
        row = {"epoch":epoch, "loss":float(np.mean(losses)), "seconds":time.time()-t0, **{f"all_{k}":v for k,v in combined.items()}}
        for domain, met in domain_metrics.items():
            row.update({f"{domain}_{k}":v for k,v in met.items()})
        history.append(row)
        print(row)
        if combined["macro_f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = combined["macro_f1"], epoch, 0
            torch.save({"model_state_dict":model.state_dict(), "labels":LABELS, "run_name":run_name,
                        "best_epoch":epoch, "metrics":row}, run_dir/"best.pt")
        else:
            bad_epochs += 1
        pd.DataFrame(history).to_csv(run_dir/"history.csv", index=False)
        if bad_epochs >= patience:
            print(f"Early stopping after {epoch} epochs; best epoch={best_epoch}, macro-F1={best_f1:.4f}")
            break
    return pd.DataFrame(history), best_f1

## 11. First smoke test — load exactly one Phase 1 model

Write one loader per architecture. Example below is a template only; adapt it to the exact class and state-dict format used in your Phase 1 training notebook. Do not run training until this loads strictly and produces `[batch, seq, hidden]`.

In [ ]:
def load_transformer_backbone(checkpoint_path):
    # TEMPLATE ONLY. Replace with your actual Phase 1 config/model class and checkpoint structure.
    # Example for a GPT-2-style Phase 1 checkpoint:
    #
    # from transformers import GPT2Config, GPT2Model
    # ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    # cfg = GPT2Config(
    #     vocab_size=32000, n_positions=4096, n_ctx=4096,
    #     n_embd=768, n_layer=12, n_head=12
    # )
    # model = GPT2Model(cfg)
    # state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt))
    # # If checkpoint is an LM-head checkpoint, filter/rename keys appropriately;
    # # never use strict=False without printing and reviewing missing/unexpected keys.
    # model.load_state_dict(state, strict=True)
    # return model
    raise NotImplementedError("Connect this to the exact Phase 1 checkpoint loader.")

# Required preflight:
# 1) configure PHASE1['transformer']['checkpoint'] and loader.
# 2) load backbone.
# 3) print missing/unexpected keys if your architecture uses key conversion.
# 4) run one dummy forward pass and verify hidden dimensions.

# TPU starter configuration.
# Note: this is not sufficient by itself to make every architecture TPU-compatible.
ARCH = "transformer"  # choose one configured architecture
BATCH_SIZE = 4
EPOCHS = 3
GRAD_ACCUM = 4
MAX_LENGTH = 512

if not TPU_AVAILABLE:
    print("TPU not detected. You can still validate data on CPU, but this is not TPU training.")
else:
    print("TPU detected. Before full training, test that your model and every operation support XLA.")

cfg = PHASE1[ARCH]
assert cfg["checkpoint"] and Path(cfg["checkpoint"]).exists(), "Set a valid Phase 1 checkpoint path in Google Drive."
assert callable(cfg["loader"]), "Define a real loader function for this architecture first."
assert tokenizer is not None, "Set PHASE1_TOKENIZER_PATH to your Phase 1 tokenizer."

backbone = cfg["loader"](cfg["checkpoint"])
hidden_size = infer_hidden_size(backbone)
model = RhetoricalClassifier(backbone, hidden_size, len(LABELS)).to(device)
train_loader, train_ds = make_loader(train_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
dev_cl_loader, _ = make_loader(dev_cl_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=False)
dev_it_loader, _ = make_loader(dev_it_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

b = next(iter(train_loader))
with torch.no_grad():
    out = model(b["input_ids"].to(device), b["attention_mask"].to(device), b["target_mask"].to(device))
print("smoke logits shape:", tuple(out.shape), "| expected:", (len(b["label_id"]), len(LABELS)))
assert out.shape[-1] == len(LABELS)

# Do not start full training until train_one_run is XLA-aware.
# Standard PyTorch GradScaler/autocast and optimizer stepping often need changes for TPU.
# For TPU, each training step must use xm.optimizer_step(optimizer) (or the equivalent
# supported by your torch_xla version), and data/model operations must stay XLA-compatible.
# history, best = train_one_run(...)


In [ ]:
# First run on a TPU: keep the micro-batch small to avoid out-of-memory errors.
# Increase only after the smoke test succeeds and TPU memory usage looks safe.
ARCH = "transformer"  # choose one configured architecture
BATCH_SIZE = 2
EPOCHS = 3
GRAD_ACCUM = 8
MAX_LENGTH = 512  # TPU-aware starting point; reduce to 256 if you hit TPU out-of-memory.

cfg = PHASE1[ARCH]
assert cfg["checkpoint"] and Path(cfg["checkpoint"]).exists(), "Set a valid Phase 1 checkpoint path in Google Drive."
assert callable(cfg["loader"]), "Define a real loader function for this architecture first."
assert tokenizer is not None, "Set PHASE1_TOKENIZER_PATH to your Phase 1 tokenizer."

backbone = cfg["loader"](cfg["checkpoint"])
hidden_size = infer_hidden_size(backbone)
model = RhetoricalClassifier(backbone, hidden_size, len(LABELS)).to(device)
train_loader, train_ds = make_loader(train_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
dev_cl_loader, _ = make_loader(dev_cl_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=False)
dev_it_loader, _ = make_loader(dev_it_rows, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

# Smoke test before training.
b = next(iter(train_loader))
with torch.no_grad():
    out = model(b["input_ids"].to(device), b["attention_mask"].to(device), b["target_mask"].to(device))
print("smoke logits shape:", tuple(out.shape), "| expected:", (len(b["label_id"]), len(LABELS)))
assert out.shape[-1] == len(LABELS)

# Uncomment only after the smoke test passes:
# history, best = train_one_run(
#     run_name=f"{ARCH}_fullft_context{CONTEXT_RADIUS}",
#     model=model,
#     train_loader=train_loader,
#     dev_loaders={"CL":dev_cl_loader, "IT":dev_it_loader},
#     epochs=EPOCHS, lr_backbone=2e-5, lr_head=1e-3,
#     grad_accum=GRAD_ACCUM, patience=2
# )
# print("best combined macro-F1:", best)


## 13. LoRA experiment (PEFT)

For a Hugging Face module with known target module names, PEFT can inject LoRA. The exact targets depend on architecture. For GPT-2, common projection module names include `c_attn` and `c_proj`; verify the names from `model.backbone.named_modules()` before configuring. Do not apply these target names blindly to Mamba/RMKV/RetNet.

In [ ]:
# Inspect target module names:
# print([name for name, module in backbone.named_modules() if isinstance(module, nn.Linear)][:100])

# Example only for compatible transformer modules:
# from peft import LoraConfig, get_peft_model, TaskType
# lora_cfg = LoraConfig(
#     r=8, lora_alpha=16, lora_dropout=0.05,
#     target_modules=["c_attn", "c_proj"],  # verify these exist in YOUR model
#     bias="none", task_type=TaskType.FEATURE_EXTRACTION
# )
# lora_backbone = get_peft_model(backbone, lora_cfg)
# lora_model = RhetoricalClassifier(lora_backbone, hidden_size, len(LABELS))
# print("trainable params:")
# lora_backbone.print_trainable_parameters()
#
# Then use train_one_run(..., run_name=f"{ARCH}_lora_context{CONTEXT_RADIUS}", model=lora_model, ...)

## 14. Compulsory LORMA and knowledge distillation — do not fake these

**LORMA:** implement the exact method/variant required by your course and validate it against its paper/reference implementation. Do not label ordinary LoRA as LORMA. The LoRMA paper describes multiplicative low-rank adaptation and rank-inflation variants; the specific variant and compatible projection layers must be chosen deliberately. Reference: https://exploration-lab.github.io/LoRMA/

**Knowledge distillation:** train a larger/stronger teacher from one of your Phase 1 models, then train a smaller student initialized from another Phase 1 model. Distill teacher probability distributions/logits plus the gold-label loss. The student must still start from your own Phase 1 weights; no external pretrained student.

In [ ]:
def distillation_loss(student_logits, teacher_logits, gold_ids, temperature=2.0, alpha=0.5):
    # alpha weights the soft-target loss; (1-alpha) weights supervised gold-label CE.
    soft = nn.functional.kl_div(
        nn.functional.log_softmax(student_logits / temperature, dim=-1),
        nn.functional.softmax(teacher_logits / temperature, dim=-1),
        reduction="batchmean"
    ) * (temperature ** 2)
    hard = nn.functional.cross_entropy(student_logits, gold_ids)
    return alpha * soft + (1.0-alpha) * hard

# Distillation training loop should:
# 1. load teacher from the best Phase 1-based fine-tuned checkpoint;
# 2. freeze teacher and run it in eval/no_grad;
# 3. load student from a smaller Phase 1 checkpoint;
# 4. minimize distillation_loss(student_logits, teacher_logits, gold_ids);
# 5. select the student by CL+IT development macro-F1 and log size/runtime.

## 15. Experiment plan and ledger

Six runs must be meaningfully different; changing only the learning rate or rank does not count. The official requirement also says to use all four Phase 1 architectures, include LORMA, and include knowledge distillation. Adapt this table after confirming which checkpoints and loader implementations are available.

In [ ]:
EXPERIMENT_PLAN = pd.DataFrame([
    {"id":"E1", "architecture":"Transformer", "method":"Full fine-tuning", "context":"target sentence only", "status":"planned"},
    {"id":"E2", "architecture":"Transformer", "method":"LoRA", "context":"previous + target + next", "status":"planned"},
    {"id":"E3", "architecture":"Mamba", "method":"Full or partial fine-tuning", "context":"previous + target + next", "status":"blocked until loader configured"},
    {"id":"E4", "architecture":"RMKV", "method":"LORMA (required; architecture-compatible implementation)", "context":"previous + target + next", "status":"blocked until loader and LORMA target modules confirmed"},
    {"id":"E5", "architecture":"RetNet", "method":"Distinct PEFT method (e.g. prefix/IA3 if supported)", "context":"previous + target + next", "status":"blocked until loader configured"},
    {"id":"E6", "architecture":"Student Phase 1 model", "method":"Knowledge distillation (required)", "context":"match strongest validated context setup", "status":"planned"},
])
display(EXPERIMENT_PLAN)

LEDGER_PATH = WORK_DIR / "experiment_ledger.csv"
def record_result(row):
    df = pd.DataFrame([row])
    df.to_csv(LEDGER_PATH, mode="a", header=not LEDGER_PATH.exists(), index=False)
    print("logged to", LEDGER_PATH)

## 16. Final test prediction writer

Load a selected best checkpoint and its matching architecture/tokenizer. Generate one label per sentence, in original order, for both domains. Validate all counts and labels before writing. This cell intentionally refuses to run until `predict_docs()` is implemented for the chosen architecture.

In [ ]:
@torch.no_grad()
def predict_rows(model, rows, tokenizer, batch_size=4):
    loader, ds = make_loader(rows, tokenizer, batch_size=batch_size, shuffle=False, with_labels=False)
    model.eval(); pred_ids = []
    for batch in loader:
        batch = {k:v.to(device) for k,v in batch.items()}
        logits = model(batch["input_ids"], batch["attention_mask"], batch["target_mask"])
        pred_ids.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[i] for i in pred_ids]

def predict_docs(model, docs, tokenizer, radius=1, batch_size=4):
    rows = flatten_docs(docs, radius=radius, with_labels=False)
    labels = predict_rows(model, rows, tokenizer, batch_size=batch_size)
    out = {}
    cursor = 0
    for d in docs:
        n = len(d["text"])
        out[d["id"]] = labels[cursor:cursor+n]
        cursor += n
        assert len(out[d["id"]]) == n
        assert all(y in LABELS for y in out[d["id"]])
    assert cursor == len(labels)
    return out

# After loading the chosen trained model and restoring its best checkpoint:
# cl_pred = predict_docs(best_model, CL_TEST, tokenizer, radius=CONTEXT_RADIUS)
# it_pred = predict_docs(best_model, IT_TEST, tokenizer, radius=CONTEXT_RADIUS)
# merged = {**cl_pred, **it_pred}
# assert len(merged) == len(cl_pred) + len(it_pred), "Duplicate document IDs across domains?"
# out_path = PRED_DIR / "prediction.json"
# with open(out_path, "w", encoding="utf-8") as f: json.dump(merged, f, ensure_ascii=False, indent=2)
# print("wrote", out_path)

## 17. Final pre-submission validation

Run this after writing `prediction.json`. It verifies document coverage, sentence counts, order-preserving list lengths, and exact official label names.

In [ ]:
def validate_predictions(pred_path, test_docs_by_domain):
    pred = read_json(pred_path)
    expected = {}
    for docs in test_docs_by_domain:
        for d in docs:
            if d["id"] in expected:
                raise ValueError(f"Duplicate test ID across files: {d['id']}")
            expected[d["id"]] = len(d["text"])
    if set(pred) != set(expected):
        missing, extra = set(expected)-set(pred), set(pred)-set(expected)
        raise ValueError(f"ID mismatch: missing={len(missing)}, extra={len(extra)}")
    for did, n in expected.items():
        labels = pred[did]
        if not isinstance(labels, list) or len(labels) != n:
            raise ValueError(f"{did}: expected {n} labels, got {len(labels) if isinstance(labels,list) else type(labels)}")
        bad = sorted(set(labels)-set(LABELS))
        if bad: raise ValueError(f"{did}: invalid labels {bad}")
    print(f"VALID: {len(pred)} documents; {sum(expected.values())} sentence predictions; official labels only.")

# validate_predictions(PRED_DIR/"prediction.json", [CL_TEST, IT_TEST])

## 18. Research notes that inform the design

- Legal rhetorical-role labeling is a sequence-labeling/classification problem; neighboring sentence context can improve predictions. See the SemEval-2023 sentence-sequence approach: https://aclanthology.org/2023.semeval-1.103/
- The LegalSeg work reports benefits from broader context and sequential relationships: https://aclanthology.org/2025.findings-naacl.63/
- “Mind Your Neighbours” studies analogous examples for rhetorical-role labeling, including inference-time and training-time approaches: https://aclanthology.org/2024.lrec-main.987/
- LORMA method details/reference implementation: https://exploration-lab.github.io/LoRMA/

These papers are motivation, not permission to import their pretrained weights. The capstone constraints take precedence: use your Phase 1 models and the supplied task data.